<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Deep_learning_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cell 1 — GPU check + setup

In [2]:
!nvidia-smi
import os
from pathlib import Path

ROOT = Path("/content/av_perception")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

/bin/bash: line 1: nvidia-smi: command not found


### Cell 2 — Install YOLOv8 + utils

In [3]:
!pip -q install ultralytics opencv-python imageio imageio-ffmpeg

from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### Cell 3 — Download dataset (YOLO-ready, no login)

In [11]:
from ultralytics.utils.downloads import download
from pathlib import Path
import shutil # Import shutil for rmtree

DATASETS_DIR = Path("datasets")
DATASETS_DIR.mkdir(exist_ok=True)

url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128.zip"

# Remove existing coco128 directory and zip file to ensure a clean re-download
coco128_dir = DATASETS_DIR / "coco128"
coco128_zip = DATASETS_DIR / "coco128.zip"

if coco128_dir.exists():
    shutil.rmtree(coco128_dir)
if coco128_zip.exists():
    coco128_zip.unlink()

download(url, dir=DATASETS_DIR)

Unzipping datasets/coco128.zip to /content/av_perception/datasets/coco128...: 100% ━━━━━━━━━━━━ 263/263 3.1Kfiles/s 0.1s


In [14]:
!ls datasets
!ls datasets/coco128

coco128  coco128.zip
images	labels	LICENSE  README.txt


In [15]:
from pathlib import Path
import yaml

yaml_data = {
    "path": "datasets/coco128",
    "train": "images/train2017",
    "val": "images/val2017",
    "names": {
        0: "person",
        2: "car",
        3: "motorcycle",
        5: "bus",
        7: "truck"
    }
}

yaml_path = Path("datasets/coco128/coco128.yaml")
with open(yaml_path, "w") as f:
    yaml.safe_dump(yaml_data, f, sort_keys=False)

print("Created:", yaml_path)

Created: datasets/coco128/coco128.yaml


In [16]:
!ls datasets/coco128

coco128.yaml  images  labels  LICENSE  README.txt


### Cell 4 — Inspect dataset

In [12]:
import os, yaml

print(os.listdir("datasets"))
print(os.listdir("datasets/coco128"))

with open("datasets/coco128/coco128.yaml") as f:
    data_yaml = yaml.safe_load(f)

data_yaml

['coco128.zip', 'coco128']
['labels', 'images', 'README.txt', 'LICENSE']


FileNotFoundError: [Errno 2] No such file or directory: 'datasets/coco128/coco128.yaml'

### Cell 5 — Train YOLOv8 (T4-friendly)

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="datasets/coco128/coco128.yaml",
    epochs=20,
    imgsz=640,
    batch=16,
    device=0,
    name="av_coco128",
)

### Cell 6 — Validate

In [ ]:
model = YOLO("runs/detect/road_detection/weights/best.pt")
model.val(data="road_data/data.yaml")

### Cell 7 — Download demo driving video

In [ ]:
!wget -q -O traffic.mp4 https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4

### Cell 8 — Detection + tracking + TTC overlay (AV logic)

In [ ]:
import cv2
import numpy as np

cap = cv2.VideoCapture("traffic.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    "traffic_ttc.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (W, H),
)

FOCAL = 700
REAL_HEIGHT = {"person": 1.7, "car": 1.5}
track_hist = {}
TTC_THRESH = 2.0

frame_id = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t = frame_id / fps
    results = model.track(frame, persist=True, conf=0.25, verbose=False)[0]

    if results.boxes.id is not None:
        boxes = results.boxes.xyxy.cpu().numpy()
        ids   = results.boxes.id.cpu().numpy()
        clss  = results.boxes.cls.cpu().numpy().astype(int)

        for box, tid, cls in zip(boxes, ids, clss):
            x1,y1,x2,y2 = map(int, box)
            label = model.names[cls]

            h = max(1, y2 - y1)
            dist = (REAL_HEIGHT.get(label,1.6)*FOCAL)/h

            ttc = None
            if tid in track_hist:
                d_prev, t_prev = track_hist[tid]
                v_rel = (d_prev - dist)/(t - t_prev + 1e-3)
                if v_rel > 0:
                    ttc = dist/v_rel

            track_hist[tid] = (dist,t)

            cv2.rectangle(frame,(x1,y1),(x2,y2),(0,255,255),2)
            txt = f"{label} id={int(tid)} d~{dist:.1f}m"
            if ttc and ttc < 10:
                txt += f" TTC~{ttc:.1f}s"
            cv2.putText(frame,txt,(x1,y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX,0.6,(0,255,255),2)

            if ttc and ttc < TTC_THRESH:
                cv2.putText(frame,"NEAR MISS!",
                            (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX,1.1,(0,0,255),3)

    out.write(frame)
    frame_id += 1

cap.release()
out.release()
print("Saved traffic_ttc.mp4")

### Cell 9 — Convert to GIF

In [ ]:
import imageio

reader = imageio.get_reader("traffic_ttc.mp4")
frames = [reader.get_data(i) for i in range(0,120,3)]
imageio.mimsave("traffic_ttc.gif", frames, fps=8)

print("Saved traffic_ttc.gif")

### Cell 10 — Display

In [ ]:
from IPython.display import Video, Image

Video("traffic_ttc.mp4", embed=True)
Image("traffic_ttc.gif")